# Check your setup

This notebook checks that your computer is ready for the course: Python, the course packages, Java and
Spark. Run it after creating the `bda` environment in the
[Setup guide](https://worldbank.github.io/BDA-with-Python/guides/software-setup.html).

**How to run it:** check that the kernel in the top right says **Python 3 (ipykernel)**, then choose
**Run → Run All Cells**. It takes a minute or two.

Each check prints ✅ if it passed, or ❌ with a one-line fix and a link to the relevant part of the
guides. The last cell tells you whether you're ready.

**Checks, in order:**

1. Python version (3.10 or later)
2. Course packages: pandas, numpy, matplotlib, scikit-learn, requests, BeautifulSoup, PySpark
3. Java (17 or later) and `JAVA_HOME`
4. Spark: start a local session, create a small DataFrame, write and read Parquet, stop Spark


In [ ]:
# ============================================================
# Helpers for recording and printing results
# ============================================================
import importlib
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

GUIDES = "https://worldbank.github.io/BDA-with-Python/guides/"
SETUP_ENV = GUIDES + "software-setup.html#step-4"
TROUBLESHOOT = GUIDES + "pyspark-setup.html#troubleshooting"

results = {}


def report(name, ok, detail="", fix="", link=""):
    """Record a check and print one line (plus a fix line if it failed)."""
    results[name] = ok
    if ok:
        print(f"✅ {name}" + (f": {detail}" if detail else ""))
    else:
        print(f"❌ {name}" + (f": {detail}" if detail else ""))
        if fix:
            print(f"   Fix: {fix}")
        if link:
            print(f"   See: {link}")


In [ ]:
# ============================================================
# Check 1: Python version (3.10 or later)
# ============================================================
version = platform.python_version()
report(
    "Python version",
    sys.version_info >= (3, 10),
    detail=f"Python {version} ({sys.executable})",
    fix="Start Jupyter from the course environment: run `conda activate bda`, then `jupyter lab`.",
    link=SETUP_ENV,
)


In [ ]:
# ============================================================
# Check 2: Course packages
# ============================================================
# Import name -> name to show (the pip/conda package name where it differs)
packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pyspark": "pyspark",
}

for module, label in packages.items():
    try:
        mod = importlib.import_module(module)
        report(f"Package {label}", True, detail=getattr(mod, "__version__", "installed"))
    except Exception as err:
        report(
            f"Package {label}",
            False,
            detail=f"cannot import {module} ({type(err).__name__})",
            fix="Use the bda environment (`conda activate bda` before `jupyter lab`), "
                "or recreate it with `conda env create -f environment.yml`.",
            link=SETUP_ENV,
        )


In [ ]:
# ============================================================
# Check 3: Java (17 or later) and JAVA_HOME
# ============================================================
java_ok = False
java_exe = shutil.which("java")
if java_exe is None:
    report(
        "Java",
        False,
        detail="`java` not found",
        fix="Activate the course environment (it includes Java 17): `conda activate bda`, then restart Jupyter.",
        link=TROUBLESHOOT,
    )
else:
    # `java -version` prints to stderr, e.g. openjdk version "17.0.11" or java version "1.8.0_401"
    proc = subprocess.run([java_exe, "-version"], capture_output=True, text=True)
    text = proc.stderr + proc.stdout
    match = re.search(r'version "(\d+)(?:\.(\d+))?', text)
    if match:
        major = int(match.group(1))
        if major == 1 and match.group(2):  # old scheme: 1.8 means Java 8
            major = int(match.group(2))
        java_ok = major >= 17
        report(
            "Java version",
            java_ok,
            detail=f"Java {major} ({java_exe})",
            fix="PySpark 4 needs Java 17 or later. Use the Java in the bda environment "
                "(`conda activate bda`) and make sure JAVA_HOME points to it.",
            link=TROUBLESHOOT,
        )
    else:
        report(
            "Java version",
            False,
            detail="could not read the version from `java -version`",
            fix="Run `java -version` in your terminal and compare with the troubleshooting table.",
            link=TROUBLESHOOT,
        )

java_home = os.environ.get("JAVA_HOME")
report(
    "JAVA_HOME",
    bool(java_home) and Path(java_home).is_dir(),
    detail=java_home or "not set",
    fix="Start Jupyter from an activated environment (`conda activate bda`, then `jupyter lab`); "
        "conda sets JAVA_HOME for you.",
    link=TROUBLESHOOT,
)


In [ ]:
# ============================================================
# Check 4: Spark (session, DataFrame, Parquet write and read)
# ============================================================
if not results.get("Package pyspark"):
    report("Spark", False, detail="skipped because PySpark is not installed",
           fix="Fix the PySpark package check above first.", link=SETUP_ENV)
elif not java_ok:
    report("Spark", False, detail="skipped because Java is missing or too old",
           fix="Fix the Java check above first.", link=TROUBLESHOOT)
else:
    from pyspark.sql import SparkSession

    spark = None
    stage = "starting a Spark session"
    try:
        spark = (
            SparkSession.builder
            .master("local[*]")
            .appName("bda-check-setup")
            .getOrCreate()
        )
        report("Spark session", True, detail=f"Spark {spark.version}")

        stage = "creating a DataFrame"
        df = spark.createDataFrame(
            [("Kigali", 1_745_555), ("Lilongwe", 1_122_000), ("Nairobi", 4_397_073)],
            ["city", "population"],
        )
        report("Spark DataFrame", df.count() == 3, detail=f"{df.count()} rows")

        stage = "writing and reading Parquet"
        with tempfile.TemporaryDirectory() as tmp:
            out = str(Path(tmp) / "check.parquet")
            df.write.mode("overwrite").parquet(out)
            n = spark.read.parquet(out).count()
        report("Spark Parquet write/read", n == 3, detail=f"{n} rows read back")
    except Exception as err:
        first_line = str(err).strip().splitlines()[0][:150] if str(err).strip() else type(err).__name__
        if platform.system() == "Windows" and stage == "writing and reading Parquet":
            fix = "On Windows, Spark needs winutils.exe and hadoop.dll to write files (or use WSL2)."
            link = GUIDES + "pyspark-setup.html#windows-winutils"
        else:
            fix = "Look up this error in the troubleshooting table."
            link = TROUBLESHOOT
        report("Spark", False, detail=f"failed while {stage}: {first_line}", fix=fix, link=link)
    finally:
        if spark is not None:
            spark.stop()


In [ ]:
# ============================================================
# Summary
# ============================================================
failed = [name for name, ok in results.items() if not ok]
print()
if not failed:
    print("🎉 You're ready for the course! Every check passed.")
else:
    print(f"❌ {len(failed)} check(s) failed:")
    for name in failed:
        print(f"   - {name}")
    print()
    print("Fix the first one (its fix is printed above), then run this notebook again: Run → Run All Cells.")
    print(f"Still stuck? See {TROUBLESHOOT}")
